In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
import xgboost as xgb

In [ ]:
### Create train dataframe
train = pd.read_csv('train.csv')

In [ ]:
### Create overall features without Id and targets
train['target'] = np.log1p(train["SalePrice"])
features = [c for c in train.columns if c not in['Id','SalePrice', 'target']]

In [ ]:
### Create lists containing the numerical and categorical columns
cats = []
nums = []
for c, dtype in zip(train[features].columns, train[features].dtypes):
    if dtype == 'object':
        cats.append(c)
    else:
        nums.append(c)

In [ ]:
### Impute means for all nulls in numerical features
from sklearn.preprocessing import Imputer
imputer = Imputer()
imputer = imputer.fit(train[nums])
train[nums] = imputer.transform(train[nums])

In [ ]:
### Calculate skewness of numerical features and transforms
from scipy.stats import skew
skewed_feats = train[nums].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
skewed_feats = skewed_feats.index
train[skewed_feats] = np.log1p(train[skewed_feats])

In [ ]:
### Scale numerical features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
train[nums] = scaler.fit_transform(train[nums])

In [ ]:
### Add dummy variables for categories
train = pd.get_dummies(train, columns = cats)

In [ ]:
### Recreate overall features without Id and targets
features = [c for c in train.columns if c not in['Id','SalePrice', 'target']]

In [ ]:
### Select ten top features (should be the same top ten from the correlation above)
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
selector = SelectKBest(f_regression, k=10).fit(train[features], train['target'])
feat_scores = [(a,t) for a, t, z in zip(features, selector.scores_, selector.get_support()) if z] 
for i in  feat_scores: print i[0], i[1]

In [ ]:
### Take the features selected by SelectKBest
top_ten_features = [a for a, t in zip(features, selector.get_support()) if t]

In [ ]:
### Root Mean Squared Log Error function for GridSearch
from sklearn.metrics import make_scorer
def rmsle(predicted, actual):
    assert(len(predicted) == len(actual))
    p = np.log(np.array(predicted) + 1)
    a = np.log(np.array(actual) + 1)
    return (((p - a)**2).sum() / len(predicted))**0.5
rmsle_loss = make_scorer(rmsle, greater_is_better=False)

In [ ]:
### Split into train and test
X_train, X_test, y_train, y_test = train_test_split(train[top_ten_features], train['target'], test_size=0.30, random_state=42)

In [ ]:
%%time
### GridSearch for number of estimators
from sklearn.model_selection import GridSearchCV
boost = xgb.XGBRegressor()
n_estimators = [50, 100, 200, 300]
tuned_parameters = dict(n_estimators=n_estimators)
grid = GridSearchCV(boost, tuned_parameters, cv=3, scoring=rmsle_loss, n_jobs=-1, verbose=2)
grid.fit(train[top_ten_features], train['target'])

In [ ]:
### summarize results
print("Best: %f using %s" % (grid.best_score_, grid.best_params_))
means = grid.cv_results_['mean_test_score']
stds = grid.cv_results_['std_test_score']
params = grid.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
  print("%f (%f) with: %r" % (mean, stdev, param))

In [ ]:
### plot results
plt.errorbar(n_estimators, means, yerr=stds)
plt.legend()
plt.xlabel('n_estimators')
plt.ylabel('Mean Squared Error')
plt.show()

In [ ]:
%%time
### GridSearch for max depth
boost = xgb.XGBRegressor(n_estimators=100)
max_depth = [3,4,5,6,7,8,9,10]
tuned_parameters = dict(max_depth=max_depth)
grid = GridSearchCV(boost, tuned_parameters, cv=3, scoring=rmsle_loss, n_jobs=-1, verbose=2)
grid.fit(train[top_ten_features], train['target'])

In [ ]:
# summarize results
print("Best: %f using %s" % (grid.best_score_, grid.best_params_))
means = grid.cv_results_['mean_test_score']
stds = grid.cv_results_['std_test_score']
params = grid.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
  print("%f (%f) with: %r" % (mean, stdev, param))

In [ ]:
plt.errorbar(max_depth, means, yerr=stds)
plt.legend()
plt.xlabel('max_depth')
plt.ylabel('Mean Squared Error')
plt.show()

In [ ]:
%%time
### GridSearch for learning rate
from sklearn.model_selection import GridSearchCV
boost = xgb.XGBRegressor(n_estimators=100, max_depth=3)
learning_rate = [0.0001, 0.001, 0.01, 0.1, 0.2, 0.3]
tuned_parameters = dict(learning_rate=learning_rate)
grid = GridSearchCV(boost, tuned_parameters, cv=3, scoring=rmsle_loss, n_jobs=-1, verbose=2)
grid.fit(train[top_ten_features], train['target'])

In [ ]:
# summarize results
print("Best: %f using %s" % (grid.best_score_, grid.best_params_))
means = grid.cv_results_['mean_test_score']
stds = grid.cv_results_['std_test_score']
params = grid.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
  print("%f (%f) with: %r" % (mean, stdev, param))

In [ ]:
plt.errorbar(learning_rate, means, yerr=stds)
plt.legend()
plt.xlabel('learning_rate')
plt.ylabel('Mean Squared Error')
plt.show()

In [ ]:
%%time
### Create dmatrix for xgboost
d_train = xgb.DMatrix(X_train, label=y_train)
d_test = xgb.DMatrix(X_test, label=y_test)

In [ ]:
%%time
### Train until model hasn't improved for 100 rounds
params = {}
params['objective'] = 'reg:linear'
params['eval_metric'] = 'rmse'
params['max_depth'] = 3
params['learning_rate'] = 0.1

watchlist = [(d_train, 'train'), (d_test, 'test')]
boost = xgb.train(params, d_train, 10000, watchlist, early_stopping_rounds=100, verbose_eval=10)

In [ ]:
%%time
### Prepare test data for predictions
test = pd.read_csv('test.csv')
test[nums] = imputer.transform(test[nums])
test[skewed_feats] = np.log1p(test[skewed_feats])
test[nums] = scaler.fit_transform(test[nums])
test = pd.get_dummies(test, columns = cats)
d_test = xgb.DMatrix(test[top_ten_features])

In [ ]:
### Make prediction and convert prediction from log
preds = np.expm1(boost.predict(d_test))

In [ ]:
### Create submission file
submission = pd.DataFrame()
submission['Id'] = test.Id
submission['SalePrice'] = preds
submission.to_csv('submission_xgb.csv', index=False)